# DATA CLEANING FOR 2012 (REMOVES 2016)
## GOAL
### The goal of this notebook is to filter the data to make it usable for all models

In [2]:
import pandas as pd
import numpy as np

### Was wir alles filtern wollen
 - [x] **time for survey taking** ->20/ 30 minutes -> 230 questions, abt 6 questions per minute, also filters just clicking.
       **NB: I set the treshold to 15 mins - Casi**
- filter rows with >20% NA --> filters out follow up questions
 --> Here we also filter all PARTY_AGENDAS-Questions
- **dont know** -> na/ filetern/ delete ppl that answer a lot of dk, row > 5% dont know weg
  **; there are 84 columns that contain at least 1 don't know; I filtered people out with 40%+ DK answers (threshold number we can still discuss! - Linus** 
- what about ppl that dont take the survey seriously and just click the **answers with patterns**? do we just accept it? can we filter this?
  - check for many same answers in ordinal questions and potentially clean them (maybe we leave it and just clean the rows that filled out the survey too fast)
  -   --> I think maybe it's better to not filter this, because if someone answers in a pattern, the randomization from the survey itself should handle this bias. I mean we still get "true" data just with higher vatiance (or noise). ANd also again I think with the 15min time-threshold this problem is mostly solved.- Linus 

- does democrat/ republican ratio change before/after cleaning #dat["PARTY_AGENDAS_rand_2016"].value_counts()
- how much data do we have left?
- data needs to work for all models -> maybe no na at all -> **dropna** #rows_with_na = dat.isna().any(axis=1).sum()
- RIEKE
The output of cleaning round 1 is found in **data/votersurvey_cleaned.parquet**

### cleaning round 2

- remove zipcodes (izip_2016)since there are almost as many zipcodes as participants 
- remove variable "PARTY_AGENDAS_rand_2016", which we had previously regardes as target variable! This variable is actually only a randomisation!
- new target variable: 'presvote16post_2016'
- remove rows that did not vote for either trump or clinton
- potentialy remove variables that ask if you favor this or this person to focus on variables asking for an attitude
The output of cleaning round 1 is found in **data/votersurvey_cleaned_noDropNA.parquet**
**NB: the NA's need to be dropped in the modeling notebooks**

### after cleaning
- pca (nice plot for the presentation, potentially aids in feature selection)
- feature selection -> correlation again (project (2) code file)
- basically all the tests and models 
- modelle fitten: random forest, deep neural network, kNN?

1. data cleaning
2. feature selection
3. model creation and training
4. repeat

In [3]:
#read data with correct NA values
df = pd.read_csv("data/VOTER_Survey_December16_Release1.csv")
dat = df
dat.replace('__NA__', np.nan, inplace=True) #correctly read in NA values
print(dat.shape)

dat.head()

/tmp/ipykernel_1710/3156404074.py:2: DtypeWarning: Columns (0: votereg_fnd_baseline, 1: religpew_muslim_baseline) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/VOTER_Survey_December16_Release1.csv")


(8000, 668)


,case_identifier,weight,PARTY_AGENDAS_rand_2016,pp_primary16_2016,pp_demprim16_2016,pp_repprim16_2016,inputstate_2016,izip_2016,votereg2_2016,votereg_f_2016,...,post_HouseCand3Name_2012,post_HouseCand3Party_2012,post_SenCand1Name_2012,post_SenCand1Party_2012,post_SenCand2Name_2012,post_SenCand2Party_2012,post_SenCand3Name_2012,post_SenCand3Party_2012,starttime_2016,endtime_2016
0,779,0.358213,Republican Party,In the Democratic primary,Hillary Clinton,NaN,California,94952,Yes,Yes,...,NaN,NaN,Shelley Berkley,Democratic,Dean Heller,Republican,NaN,NaN,29nov2016 22:59:43,29nov2016 23:28:24
1,2108,0.562867,Republican Party,In the Republican primary,NaN,Donald Trump,Arizona,85298,Yes,Yes,...,NaN,NaN,Richard Carmona,Democratic,Jeff Flake,Republican,NaN,NaN,29nov2016 15:41:28,29nov2016 18:58:28
2,2597,0.552138,Republican Party,In the Democratic primary,Hillary Clinton,NaN,Wisconsin,54904,Yes,Yes,...,NaN,NaN,Tammy Baldwin,Democratic,Tommy Thompson,Republican,NaN,NaN,29nov2016 16:08:39,29nov2016 16:32:43
3,4148,0.207591,Democratic Party,In the Democratic primary,Someone else,NaN,Oklahoma,74104,Yes,Yes,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14dec2016 18:46:33,14dec2016 19:11:20
4,4460,0.333729,Republican Party,In the Republican primary,NaN,Marco Rubio,Texas,78253,Yes,Yes,...,NaN,NaN,Paul Sadler,Democratic,Ted Cruz,Republican,NaN,NaN,01dec2016 10:17:47,01dec2016 10:59:48


### remove specific columns and rows

In [4]:
#remove columns we don't need, recognized by patterns: 2012, baseline, rnd

#initial shape
print("initial shape: ", dat.shape)

# REMOVE ALL 2016 COUMNS, ONLY KEEP 2012 AND BASELINE
#Remove columns that contain "2012" in their names
dat = dat.loc[:, ~dat.columns.str.contains('2016')].copy()
print("drop 2016: ", dat.shape) #check how many rows remain

#REMOVES TOO MANY COLUMNS (ONLY 16 ISH REMAIN IN THE END INSTEAD OF 268)
#remove columns that contain 'baseline' in their name (older than 2012)
#dat = dat.loc[:, ~dat.columns.str.contains('baseline')].copy()
#print("drop baseline: ", dat.shape) #check how many rows remain 

#remove columns that contain 'rnd' in their name (randomisation order, not a question)
dat = dat.loc[:, ~dat.columns.str.contains('rnd')].copy()
print("drop randomistion orders: ", dat.shape) #check how many rows remain

# Exclude columns with more than 20% NA: for the follow up question
perc = 0.20
threshold = len(dat) * perc
dat = dat.loc[:, dat.isnull().sum() <= threshold]

print(f"drop columns with more than {perc*100}% NA values: ", dat.shape) #see final number of rows


initial shape:  (8000, 668)
drop 2016:  (8000, 396)
drop randomistion orders:  (8000, 390)
drop columns with more than 20.0% NA values:  (8000, 268)


In [5]:
dat["presvote16post_2016"] = df["presvote16post_2016"]

In [6]:
#remove all rows that did not vote for either clinton or trump
dat = dat[dat["presvote16post_2016"].isin(['Donald Trump', 'Hillary Clinton'])]

In [10]:
"""
#drop columns that ask how favorable your opinion towards a person is
#or who of two person you prefer
#in order to only have columns asking for attitudes

#open question: drop inputstate as well? --> would leave it - Linus
cols_to_drop =[
    "fav_trump_2016",
    "fav_cruz_2016",
    "fav_ryan_2016",
    "fav_romn_2016",
    "fav_obama_2016",
    "fav_hrc_2016",
    "fav_sanders_2016",
    "fav_rubio_2016", 
    "Clinton_Rubio_2016",
    "Clinton_Cruz_2016",
    "Sanders_Trump_2016",
    "Sanders_Rubio_2016",
    "pid7_2016" # same question as pid3_2016
]

dat = dat.drop(columns = cols_to_drop)
"""

'\n#drop columns that ask how favorable your opinion towards a person is\n#or who of two person you prefer\n#in order to only have columns asking for attitudes\n\n#open question: drop inputstate as well? --> would leave it - Linus\ncols_to_drop =[\n    "fav_trump_2016",\n    "fav_cruz_2016",\n    "fav_ryan_2016",\n    "fav_romn_2016",\n    "fav_obama_2016",\n    "fav_hrc_2016",\n    "fav_sanders_2016",\n    "fav_rubio_2016", \n    "Clinton_Rubio_2016",\n    "Clinton_Cruz_2016",\n    "Sanders_Trump_2016",\n    "Sanders_Rubio_2016",\n    "pid7_2016" # same question as pid3_2016\n]\n\ndat = dat.drop(columns = cols_to_drop)\n'

In [7]:
dat["starttime_2016"] = df["starttime_2016"]
dat["endtime_2016"] = df["endtime_2016"]

In [31]:
#calculate time for filling out survey and clean specific columns afterwards

#calculate time for filling out survey
dat["starttime_2016"] = pd.to_datetime(dat["starttime_2016"], format = "%d%b%Y %H:%M:%S") #format 
dat["endtime_2016"] = pd.to_datetime(dat["endtime_2016"], format = "%d%b%Y %H:%M:%S") #format

dat["time_for_survey"] = dat["endtime_2016"] - dat["starttime_2016"] #timedelta

rows_to_drop = dat["time_for_survey"] < pd.Timedelta(15, "minutes")
print("No of rows to drop: ", np.sum(rows_to_drop))#20 minutes kicks out 2700 rows, 15 minutes only 1000 rows
dat = dat[~rows_to_drop] #drop rows

# clean specific columns
dat = dat.drop(["case_identifier",  "weight", "starttime_2016", "endtime_2016", "time_for_survey"], axis = 1) #envpoll2_2016 has many and almost exclusively trump voters that did not answer 

#print result
print("dat shape after dropping participants that took too long to fill out survey: ", dat.shape)

No of rows to drop:  0
dat shape after dropping participants that took too long to fill out survey:  (6014, 267)


In [32]:
# Don't know filtering

dk_cols = []
for col in dat.columns:
    if dat[col].isin(["Don't know"]).any():
        dk_cols.append(col)

print(f"{len(dk_cols)} columns contain 'Don't know'")
#print(dk_cols)

# How many people have how many DK in total (from 0 to 79 DK / person
dk_per_person = (dat[dk_cols] == "Don't know").sum(axis=1)
#print(dk_per_person.value_counts().sort_index())

# Who are the people that answered DK in more than 50% of DK columns (42+ out of 84)
threshold = 0.4
min_dk = len(dk_cols) * threshold

to_much_dk = dk_per_person[dk_per_person > min_dk]

print(f"People answering DK in more than {threshold * 100}% of don't know columns: {len(to_much_dk)}")
print(to_much_dk.sort_values(ascending=False))

# apply on data
print(f"before filtering DK candidats: {dat.shape}")
dat = dat.drop(index=to_much_dk.index)
print(f"after filtering DK candidats: {dat.shape}")

15 columns contain 'Don't know'
People answering DK in more than 40.0% of don't know columns: 0
Series([], dtype: int64)
before filtering DK candidats: (6014, 267)
after filtering DK candidats: (6014, 267)


### datetype conversions

In [17]:
"""
#convert "feeling thermometer" columns to numeric
#some values are e.g. "75 - somewhat favorable" and need to be converted to a float - hence the slightly complicated code

#NB: I replaced NaN values with the column mean for the feeling thermometer columns
#this prevents dropping 500 additional columns in the final dat.dropna()
#however, this can be discussed and we can remove this code section

# List of columns to process (feeling thermometer columns)
thermometer_cols = [
    "ft_black_2016", "ft_white_2016", "ft_hisp_2016", "ft_asian_2016",
    "ft_muslim_2016", "ft_jew_2016", "ft_christ_2016", "ft_fem_2016",
    "ft_immig_2016", "ft_blm_2016", "ft_wallst_2016", "ft_gays_2016",
    "ft_unions_2016", "ft_police_2016", "ft_altright_2016"
]
"""
# Extract numeric part and convert to float for each column
for col in thermometer_cols:
    dat[col] = pd.to_numeric(
        dat[col].astype(str).str.extract(r'(\d+)')[0],
        errors='coerce'  # Convert invalid parsing to NaN
    )
    col_mean = dat[col].mean()  # Compute column mean 
    dat[col] = dat[col].fillna(col_mean)  # Replace NaN with column mean

# Verify changes (optional)
#print(dat[thermometer_cols].head())


'\n#convert "feeling thermometer" columns to numeric\n#some values are e.g. "75 - somewhat favorable" and need to be converted to a float - hence the slightly complicated code\n\n#NB: I replaced NaN values with the column mean for the feeling thermometer columns\n#this prevents dropping 500 additional columns in the final dat.dropna()\n#however, this can be discussed and we can remove this code section\n\n# List of columns to process (feeling thermometer columns)\nthermometer_cols = [\n    "ft_black_2016", "ft_white_2016", "ft_hisp_2016", "ft_asian_2016",\n    "ft_muslim_2016", "ft_jew_2016", "ft_christ_2016", "ft_fem_2016",\n    "ft_immig_2016", "ft_blm_2016", "ft_wallst_2016", "ft_gays_2016",\n    "ft_unions_2016", "ft_police_2016", "ft_altright_2016"\n]\n\n# Extract numeric part and convert to float for each column\nfor col in thermometer_cols:\n    dat[col] = pd.to_numeric(\n        dat[col].astype(str).str.extract(r\'(\\d+)\')[0],\n        errors=\'coerce\'  # Convert invalid pars

In [33]:
#save colnames and their dtypes for inspection in a csv
df_rownames_dtypes = pd.DataFrame({'colname': pd.Series(dat.columns), 
                        'dtype': dat.dtypes.values,
           
                                   'unique_values': dat.nunique().values})
df_rownames_dtypes.to_csv("data/df_rownames_dtypes_2012.csv")

In [34]:
dat.info() #check 

<class 'pandas.DataFrame'>
Index: 6014 entries, 0 to 7999
Columns: 267 entries, newsint2_baseline to presvote16post_2016
dtypes: category(250), float64(14), int64(3)
memory usage: 2.6 MB


In [35]:
#convert all variables with dtype "str" to categories
for i in range(len(df_rownames_dtypes["colname"])):
    colname = df_rownames_dtypes["colname"][i]
    dtype = df_rownames_dtypes["dtype"][i]
    if dtype == "str":
        dat[f'{colname}'] = dat[f'{colname}'].astype('category')

#convert zip code to category
#dat['izip_2016'] = dat['izip_2016'].astype('category')

### some checks before finishing up

In [36]:
dat.info() #correct dtypes, memory usage now 10times smaller

<class 'pandas.DataFrame'>
Index: 6014 entries, 0 to 7999
Columns: 267 entries, newsint2_baseline to presvote16post_2016
dtypes: category(250), float64(14), int64(3)
memory usage: 2.6 MB


In [37]:
#update csv with final colnames and their dtypes for inspection
df_rownames_dtypes = pd.DataFrame({'colname': pd.Series(dat.columns), 
                        'dtype': dat.dtypes.values,
                                 'unique_values': dat.nunique().values})
df_rownames_dtypes.to_csv("data/df_rownames_dtypes_2012.csv")

In [23]:
dat.head()

,case_identifier,weight,newsint2_baseline,track_baseline,trustgovt_baseline,persfinretro_baseline,econtrend_baseline,fatalism2_baseline,obamaapp_baseline,watchtv_baseline,...,post_ideo5_2012,post_newsint_2012,post_HouseCand1Name_2012,post_HouseCand1Party_2012,post_HouseCand2Name_2012,post_HouseCand2Party_2012,presvote16post_2016,starttime_2016,endtime_2016,time_for_survey
0,779,0.358213,Most of the time,Generally headed in the right direction,Some of the time,About the same as now,Getting better,Disagree somewhat,Somewhat Approve,3 - 4 hours,...,Moderate,Most of the time,John Oceguera,Democratic,Joe Heck,Republican,Hillary Clinton,2016-11-29 22:59:43,2016-11-29 23:28:24,0 days 00:28:41
1,2108,0.562867,Most of the time,Off on the wrong track,Some of the time,Better off financially,About the same,Disagree strongly,Strongly Disapprove,More than 4 hours,...,Conservative,Most of the time,Spencer Morgan,Democratic,Matt Salmon,Republican,Donald Trump,2016-11-29 15:41:28,2016-11-29 18:58:28,0 days 03:17:00
2,2597,0.552138,Most of the time,Generally headed in the right direction,Most of the time,Worse off financially,Getting better,Agree strongly,Somewhat Approve,3 - 4 hours,...,Moderate,Most of the time,Joe Kallas,Democratic,Tom Petri,Republican,Hillary Clinton,2016-11-29 16:08:39,2016-11-29 16:32:43,0 days 00:24:04
4,4460,0.333729,Most of the time,Off on the wrong track,Some of the time,Worse off financially,Getting worse,Disagree somewhat,Strongly Disapprove,1 - 2 hours,...,Conservative,Most of the time,Henry Cuellar,Democratic,William Hayward,Republican,Donald Trump,2016-12-01 10:17:47,2016-12-01 10:59:48,0 days 00:42:01
5,5225,0.207186,Most of the time,Not sure,Some of the time,About the same as now,About the same,Agree somewhat,Strongly Approve,More than 4 hours,...,Very liberal,Most of the time,Anna Eshoo,Democratic,Dave Chapman,Republican,Hillary Clinton,2016-11-29 15:30:37,2016-11-29 15:50:15,0 days 00:19:38


In [24]:
""" #redundant
# Count rows with any NaN
rows_with_na = dat.isna().any(axis=1).sum()
print("rows with at least one NaN: ", rows_with_na)
print("Total number of rows: ", len(dat))
print("Number of rows without NaN: ", len(dat)- rows_with_na)
"""

' #redundant\n# Count rows with any NaN\nrows_with_na = dat.isna().any(axis=1).sum()\nprint("rows with at least one NaN: ", rows_with_na)\nprint("Total number of rows: ", len(dat))\nprint("Number of rows without NaN: ", len(dat)- rows_with_na)\n'

In [38]:
dat["presvote16post_2016"].value_counts() #the proportion should remain the same after dropping rows

presvote16post_2016
Donald Trump       3030
Hillary Clinton    2984
Name: count, dtype: int64

In [39]:
'''
#temporary
trump = (dat[dat["presvote16post_2016"] == 'Donald Trump'])#.to_csv('data/trump_2012.csv')
mask = trump.isna().any(axis = 1)
trump[mask].to_csv('data/trump_na_2012.csv')
'''

'\n#temporary\ntrump = (dat[dat["presvote16post_2016"] == \'Donald Trump\'])#.to_csv(\'data/trump_2012.csv\')\nmask = trump.isna().any(axis = 1)\ntrump[mask].to_csv(\'data/trump_na_2012.csv\')\n'

### drop all rows that contain NA values and save output to parquet

In [40]:
#drop all
dat_dropna = dat.dropna()

In [41]:
print("shape after cleaning: ", dat.shape)
print("\nProportion check of target variable")
dat_dropna["presvote16post_2016"].value_counts() #still roughly the same proportion

shape after cleaning:  (6014, 267)

Proportion check of target variable


presvote16post_2016
Hillary Clinton    750
Donald Trump       550
Name: count, dtype: int64

**NA still need to be dropped in the model notebooks!!**

In [42]:
#save cleaned df for next steps
dat.to_parquet('data/votersurvey_cleaned_noDropNA_2012.parquet')

#read the df with the following code: 
#df = pd.read_parquet('data/votersurvey_cleaned_noDropNA.parquet')
#optional: select features identified in feature selection
#drop NA's only afterwards 
#df = df.dropna() 
